In [2]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## ENA

In [3]:
ds_a = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/rlproffex1thorC1/*.nc')
ds_c = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/mwrret2turnC1/*.nc')
ds_c_h = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/arsclkazrbnd1kolliasC1/*.nc')

full_time = pd.date_range("2016-01-01", "2023-12-31 23:58:00", freq="2min")
ds_a = ds_a.reindex(time=full_time) # 由于ds_a 有缺失数据，补齐ds_a的缺失数据时间轴


### cloud height resample from 4s to 2min

In [74]:
ds = ds_c_h.sortby("time").copy()

# 若有占位值（如 -1），先屏蔽，避免参与插值
for v in ["cloud_layer_base_height", "cloud_layer_top_height"]:
    ds[v] = ds[v].where(ds[v] >= 0)          # <0 视为缺测
    ds[v] = ds[v].where(ds[v] <= 25000)      # 可选：上限


t0 = pd.to_datetime(ds.time.values[0])
t1 = pd.to_datetime(ds.time.values[-1])

start = t0 if (t0 == t0.floor("2min")) else t0.ceil("2min")
end   = t1.floor("2min")

new_time = xr.DataArray(pd.date_range(start, end, freq="2min"),
                        dims="time", name="time")

ds_2min = ds.interp(time=new_time, method="linear")

hist = ds.attrs.get("history", "")
ds_2min = ds_2min.assign_attrs({
    **ds.attrs,
    "history": (hist + "\n" if hist else "") +
               "time resampled by linear interpolation to 2min grid"
})

In [4]:
# ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/cloud_height_resample.nc')

### luquid water pathway resample

In [2]:
ds_c = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/mwrret2turnC1/*.nc')

In [3]:
import numpy as np
import pandas as pd
import xarray as xr

# ================= 参数 =================
START   = "2016-01-01 00:00:00"
END     = "2023-12-31 23:58:00"      # 2-min 频率下最后一个点
FREQ    = "2min"
MAX_GAP = np.timedelta64(10, "m")    # 原始相邻观测间隔 > 10 分钟的整段置 NaN

# ================= 准备 =================
ds_c = ds_c.sortby("time")           # 确保时间单调递增
time_src = ds_c.indexes["time"].values

# 目标 2-min 时间轴
time_2min = pd.date_range(START, END, freq=FREQ)
assert len(time_2min) == 2_103_840, f"目标点数不符: {len(time_2min)}"

# ================= 找出“大缺口”区间（>10min） =================
dt = np.diff(time_src)               # timedelta64[ns]
big = dt > MAX_GAP
gap_starts = time_src[:-1][big]
gap_ends   = time_src[1:][big]

# 在目标时间轴上构造掩膜：True=有效；False=落在大缺口内部
mask = np.ones(time_2min.shape, dtype=bool)
if big.any():
    i0 = time_2min.searchsorted(gap_starts, side="right")
    i1 = time_2min.searchsorted(gap_ends,   side="left")
    for a, b in zip(i0, i1):
        mask[a:b] = False
mask_da = xr.DataArray(mask, dims="time", coords={"time": time_2min})

# ================== 在插值之前按 QC 过滤（仅保留 qc==0） ==================
# 若 qc 中有 NaN，一律视为无效（过滤掉）
qc_da = ds_c["qc_phys_lwp"]          # (time,)
qc_good = (qc_da == 0).fillna(False) # True = 有效观测

# 仅对浮点变量做 QC 掩膜；非浮点按需另处理
float_vars = [v for v in ds_c.data_vars if np.issubdtype(ds_c[v].dtype, np.floating)]

# 先基于 QC 置 NaN（坏点丢弃），再插值到 2-min
ds_out = xr.Dataset(coords={"time": time_2min})
for v in float_vars:
    da_qc = ds_c[v].where(qc_good)   # 这里将 qc != 0 或 NaN 的点设为 NaN
    da_i  = da_qc.interp(time=time_2min, method="linear", assume_sorted=True)
    ds_out[v] = da_i.where(mask_da)  # 大缺口内全部 NaN

# 如需把整型/分类变量也带上（不插值），可以按需最近邻并同样套用“大缺口掩膜”和 QC 掩膜：
# for v in [v for v in ds_c.data_vars if v not in float_vars]:
#     da_qc = ds_c[v].where(qc_good)
#     ds_out[v] = da_qc.interp(time=time_2min, method="nearest", assume_sorted=True).where(mask_da)

# （可选）设置适度分块，控制内存
ds_out = ds_out.chunk({"time": 200_000})

# ================= 简单检查 =================
print(ds_out)
print("number of target :", ds_out.sizes["time"])
print("number of large gaps (>10min):", gap_starts.size)
print("Number of target points covered by large gaps:", int((~mask).sum()))



<xarray.Dataset> Size: 25MB
Dimensions:   (time: 2103840)
Coordinates:
  * time      (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    phys_lwp  (time) float32 8MB dask.array<chunksize=(200000,), meta=np.ndarray>
number of target : 2103840
number of large gaps (>10min): 12034
Number of target points covered by large gaps: 198979


In [5]:
ds_out

<xarray.Dataset> Size: 25MB
Dimensions:   (time: 2103840)
Coordinates:
  * time      (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    phys_lwp  (time) float32 8MB dask.array<chunksize=(200000,), meta=np.ndarray>

In [6]:
ds_out.to_netcdf("/data/ggong/ARM_monthly/ENA/ds_lwp_2min_gap10min_nan.nc")

###  Count the frequency and proportion of each time step

In [4]:
# 计算相邻时间差（单位秒）
t = ds_c.indexes["time"].values              # DatetimeIndex -> numpy datetime64[ns]
dt_s = (t[1:] - t[:-1]).astype("timedelta64[s]").astype(np.int64)

# 每个时间步长（秒）的出现次数
vc = pd.Series(dt_s).value_counts().sort_index()

# 表格：step(秒) / 可读标签 / 次数 / 占比(%)
df_steps = vc.rename_axis("step_s").reset_index(name="count")
df_steps["step_label"] = pd.to_timedelta(df_steps["step_s"], unit="s").astype(str)
df_steps["share_%"] = df_steps["count"] / df_steps["count"].sum() * 100

print("=== Statistics of each time step (seconds) ===")
print(df_steps.tail(20))          # 先看前20行
print("Number of unique steps:", len(df_steps))
print("Total number of intervals:", int(df_steps["count"].sum()))

=== Statistics of each time step (seconds) ===
       step_s  count        step_label   share_%
1592    49871      1   0 days 13:51:11  0.000001
1593    55221      1   0 days 15:20:21  0.000001
1594    56519      1   0 days 15:41:59  0.000001
1595    63158      1   0 days 17:32:38  0.000001
1596    69909      1   0 days 19:25:09  0.000001
1597    72787      1   0 days 20:13:07  0.000001
1598    77123      1   0 days 21:25:23  0.000001
1599    86402      1   1 days 00:00:02  0.000001
1600    86404      1   1 days 00:00:04  0.000001
1601    86430      1   1 days 00:00:30  0.000001
1602    88676      1   1 days 00:37:56  0.000001
1603    88773      1   1 days 00:39:33  0.000001
1604   120766      1   1 days 09:32:46  0.000001
1605   156546      1   1 days 19:29:06  0.000001
1606   172802      2   2 days 00:00:02  0.000003
1607   193930      1   2 days 05:52:10  0.000001
1608   262833      1   3 days 01:00:33  0.000001
1609   621963      1   7 days 04:46:03  0.000001
1610  1641675      1  

## SGP

In [3]:
ds_a = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/rlproffex1thorC1/*.nc')
ds_c = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/mwrret2turnC1/*.nc')
ds_c_h = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/arsclkazrbnd1kolliasC1/*.nc')

full_time = pd.date_range("2016-01-01", "2023-12-31 23:58:00", freq="2min")
ds_a = ds_a.reindex(time=full_time) # 由于ds_a 有缺失数据，补齐ds_a的缺失数据时间轴

In [4]:
ds = ds_c_h.sortby("time").copy()

# 若有占位值（如 -1），先屏蔽，避免参与插值
for v in ["cloud_layer_base_height", "cloud_layer_top_height"]:
    ds[v] = ds[v].where(ds[v] >= 0)          # <0 视为缺测
    ds[v] = ds[v].where(ds[v] <= 25000)      # 可选：上限


t0 = pd.to_datetime(ds.time.values[0])
t1 = pd.to_datetime(ds.time.values[-1])

start = t0 if (t0 == t0.floor("2min")) else t0.ceil("2min")
end   = t1.floor("2min")

new_time = xr.DataArray(pd.date_range(start, end, freq="2min"),
                        dims="time", name="time")

ds_2min = ds.interp(time=new_time, method="linear")

hist = ds.attrs.get("history", "")
ds_2min = ds_2min.assign_attrs({
    **ds.attrs,
    "history": (hist + "\n" if hist else "") +
               "time resampled by linear interpolation to 2min grid"
})

In [5]:
ds_2min

<xarray.Dataset> Size: 185MB
Dimensions:                  (time: 2103840, layer: 10)
Coordinates:
  * layer                    (layer) int32 40B 0 1 2 3 4 5 6 7 8 9
  * time                     (time) datetime64[ns] 17MB 2016-01-01 ... 2023-1...
Data variables:
    cloud_layer_base_height  (time, layer) float32 84MB dask.array<chunksize=(2103840, 10), meta=np.ndarray>
    cloud_layer_top_height   (time, layer) float32 84MB dask.array<chunksize=(2103840, 10), meta=np.ndarray>
Attributes: (12/18)
    command_line:                     idl -R -n kazrarsclc0 -s sgp -f C1 -b 2...
    Conventions:                      ARM-1.2
    process_version:                  vap-kazrarscl-1.0.0-devel
    dod_version:                      arsclkazrbnd1kollias-c0-1.0
    site_id:                          sgp
    platform_id:                      arsclkazrbnd1kollias
    ...                               ...
    radar_modes_in_use:               N/A
    radar_operating_frequency_burst:  N/A
    radar_operating_frequency_chirp:  N/A
    comment:                          Reflectivity values have not yet been c...
    doi:                              10.5439/1393438
    history:                          created by user ttoto on machine talc.d...

In [6]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/cloud_height_resample.nc')

#### luquid water path -- SGP

In [7]:
ds_c = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/mwrret2turnC1/*.nc')

In [8]:
import numpy as np
import pandas as pd
import xarray as xr

# ================= 参数 =================
START   = "2016-01-01 00:00:00"
END     = "2023-12-31 23:58:00"      # 2-min 频率下最后一个点
FREQ    = "2min"
MAX_GAP = np.timedelta64(10, "m")    # 原始相邻观测间隔 > 10 分钟的整段置 NaN

# ================= 准备 =================
ds_c = ds_c.sortby("time")           # 确保时间单调递增
time_src = ds_c.indexes["time"].values

# 目标 2-min 时间轴
time_2min = pd.date_range(START, END, freq=FREQ)
assert len(time_2min) == 2_103_840, f"目标点数不符: {len(time_2min)}"

# ================= 找出“大缺口”区间（>10min） =================
dt = np.diff(time_src)               # timedelta64[ns]
big = dt > MAX_GAP
gap_starts = time_src[:-1][big]
gap_ends   = time_src[1:][big]

# 在目标时间轴上构造掩膜：True=有效；False=落在大缺口内部
mask = np.ones(time_2min.shape, dtype=bool)
if big.any():
    i0 = time_2min.searchsorted(gap_starts, side="right")
    i1 = time_2min.searchsorted(gap_ends,   side="left")
    for a, b in zip(i0, i1):
        mask[a:b] = False
mask_da = xr.DataArray(mask, dims="time", coords={"time": time_2min})

# ================== 在插值之前按 QC 过滤（仅保留 qc==0） ==================
# 若 qc 中有 NaN，一律视为无效（过滤掉）
qc_da = ds_c["qc_phys_lwp"]          # (time,)
qc_good = (qc_da == 0).fillna(False) # True = 有效观测

# 仅对浮点变量做 QC 掩膜；非浮点按需另处理
float_vars = [v for v in ds_c.data_vars if np.issubdtype(ds_c[v].dtype, np.floating)]

# 先基于 QC 置 NaN（坏点丢弃），再插值到 2-min
ds_out = xr.Dataset(coords={"time": time_2min})
for v in float_vars:
    da_qc = ds_c[v].where(qc_good)   # 这里将 qc != 0 或 NaN 的点设为 NaN
    da_i  = da_qc.interp(time=time_2min, method="linear", assume_sorted=True)
    ds_out[v] = da_i.where(mask_da)  # 大缺口内全部 NaN

# 如需把整型/分类变量也带上（不插值），可以按需最近邻并同样套用“大缺口掩膜”和 QC 掩膜：
# for v in [v for v in ds_c.data_vars if v not in float_vars]:
#     da_qc = ds_c[v].where(qc_good)
#     ds_out[v] = da_qc.interp(time=time_2min, method="nearest", assume_sorted=True).where(mask_da)

# （可选）设置适度分块，控制内存
ds_out = ds_out.chunk({"time": 200_000})

# ================= 简单检查 =================
print(ds_out)
print("number of target :", ds_out.sizes["time"])
print("number of large gaps (>10min):", gap_starts.size)
print("Number of target points covered by large gaps:", int((~mask).sum()))

<xarray.Dataset> Size: 25MB
Dimensions:   (time: 2103840)
Coordinates:
  * time      (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    phys_lwp  (time) float32 8MB dask.array<chunksize=(200000,), meta=np.ndarray>
number of target : 2103840
number of large gaps (>10min): 3269
Number of target points covered by large gaps: 432429


In [9]:
ds_out

<xarray.Dataset> Size: 25MB
Dimensions:   (time: 2103840)
Coordinates:
  * time      (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    phys_lwp  (time) float32 8MB dask.array<chunksize=(200000,), meta=np.ndarray>

In [10]:
ds_out.to_netcdf("/data/ggong/ARM_monthly/SGP/ds_lwp_2min_gap10min_nan.nc")

In [14]:
# 计算相邻时间差（单位秒）
t = ds_c.indexes["time"].values              # DatetimeIndex -> numpy datetime64[ns]
dt_s = (t[1:] - t[:-1]).astype("timedelta64[s]").astype(np.int64)

# 每个时间步长（秒）的出现次数
vc = pd.Series(dt_s).value_counts().sort_index()

# 表格：step(秒) / 可读标签 / 次数 / 占比(%)
df_steps = vc.rename_axis("step_s").reset_index(name="count")
df_steps["step_label"] = pd.to_timedelta(df_steps["step_s"], unit="s").astype(str)
df_steps["share_%"] = df_steps["count"] / df_steps["count"].sum() * 100

print("=== Statistics of each time step (seconds) ===")
print(df_steps.head(20))          # 先看前20行
print("Number of unique steps:", len(df_steps))
print("Total number of intervals:", int(df_steps["count"].sum()))

=== Statistics of each time step (seconds) ===
    step_s     count       step_label    share_%
0        1  50854101  0 days 00:00:01  79.505857
1        2   1263426  0 days 00:00:02   1.975254
2        3       536  0 days 00:00:03   0.000838
3        4       658  0 days 00:00:04   0.001029
4        5      5437  0 days 00:00:05   0.008500
5        6      1299  0 days 00:00:06   0.002031
6        7       567  0 days 00:00:07   0.000886
7        8       956  0 days 00:00:08   0.001495
8        9      2229  0 days 00:00:09   0.003485
9       10      2848  0 days 00:00:10   0.004453
10      11   9181571  0 days 00:00:11  14.354568
11      12   1952410  0 days 00:00:12   3.052419
12      13       267  0 days 00:00:13   0.000417
13      14    204120  0 days 00:00:14   0.319123
14      15    214355  0 days 00:00:15   0.335125
15      16       199  0 days 00:00:16   0.000311
16      17         6  0 days 00:00:17   0.000009
17      18         8  0 days 00:00:18   0.000013
18      19        62  